# 1장 02. 결측, 범위, label 확인

이 notebook은 평가 데이터에서 가장 먼저 볼 세 가지를 확인합니다. 결측, 값 범위, label 분포입니다.


## 기본 개념: 결측, 범위, label 품질

결측값은 값이 비어 있는 상태이고, 범위 오류는 값은 있지만 의미 있는 구간을 벗어난 상태입니다. MLOps에서는 둘 다 데이터 계약 위반 후보입니다. 모델 코드는 실행될 수 있어도 입력 품질이 흔들리면 score와 prediction이 달라질 수 있습니다.

Label 품질은 모델 평가의 기준을 결정합니다. Feature 품질이 낮으면 모델 입력이 흔들리고, label 품질이 낮으면 metric이 흔들립니다. 이 둘을 구분하지 않으면 모델 성능 문제와 데이터 생성 문제를 섞어서 보게 됩니다.

운영 파이프라인에서는 결측, 범위, label 확인이 수동 탐색으로 끝나지 않아야 합니다. Great Expectations 같은 검증 도구, 학습 pipeline의 validation step, 배포 전 gate, 운영 request validation이 같은 규칙을 공유해야 합니다.

| 확인 항목 | 흔한 원인 | MLOps에서 연결되는 단계 |
| --- | --- | --- |
| Feature 결측 | 수집 누락, join 실패, 전처리 오류 | data validation, feature pipeline |
| 범위 오류 | 단위 변경, 센서/클라이언트 오류, parsing 실패 | validation rule, request schema |
| Label 이상 | 매핑 오류, 기준 변경, annotation 문제 | evaluation, model selection |
| Class 분포 변화 | sampling 변화, case mix 변화 | monitoring, drift review |

이 노트북의 목적은 모든 원인을 찾는 것이 아닙니다. 뒤 단계의 모델 metric을 읽기 전에 “데이터 자체가 평가 근거로 쓸 수 있는 상태인가”를 작은 표로 확인하는 것입니다.


In [ ]:
import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
aiq = prepared.aiq_lite


In [ ]:
dataframe, source = aiq.load_csv_or_sample(
    "data/vital_signs_evaluation_baseline.csv",
    aiq.sample_vital_signs(),
    nrows=200,
)
print("source:", source)
print("rows:", len(dataframe))


## 1. 결측 확인

필수 컬럼별로 비어 있는 값이 몇 개인지 봅니다.


In [ ]:
missing = dataframe[aiq.REQUIRED_COLUMNS].isna().sum()
missing.to_frame(name="missing_count")


## 2. 범위 확인

feature 값이 정한 범위를 벗어나는지 봅니다. 범위 밖 값이 있으면 어느 컬럼인지 먼저 확인합니다.


In [ ]:
rows = []
for column, limits in aiq.VALID_RANGES.items():
    if column in dataframe.columns:
        low, high = limits
        values = pd.to_numeric(dataframe[column], errors="coerce")
        out_count = ((values < low) | (values > high)).sum()
        rows.append({"column": column, "out_of_range": int(out_count)})

pd.DataFrame(rows)


## 3. label 분포 확인

정답 label이 `high_risk`와 `low_risk`로 들어 있는지 확인합니다.


In [ ]:
labels = dataframe["label"].map(aiq.normalize_label)
labels.value_counts(dropna=False).to_frame(name="count")
